In [1]:
# 1. Nạp dữ liệu gốc, kiểm tra schema và audit
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
from src.utils import audit, check_disguised_missing
from src.pipeline import load_raw_data, clean_transactions

df_raw = load_raw_data(ROOT / "data/raw")
print(f"Kích thước dữ liệu gốc: {df_raw.shape[0]:,} dòng, {df_raw.shape[1]} cột")
display(audit(df_raw).sort_values("pct_missing", ascending=False))
display(check_disguised_missing(df_raw))

Nguồn: online_retail_II.csv | 1,067,371 dòng
Kích thước dữ liệu gốc: 1,067,371 dòng, 8 cột


,dtype,n_missing,pct_missing,n_unique,sample
Customer ID,float64,243007,22.77,5942,13085.0
Description,str,4382,0.41,5698,15CM CHRISTMAS GLASS BALL 20 LIGHTS
StockCode,str,0,0.00,5305,85048
Invoice,str,0,0.00,53628,489434
Quantity,int64,0,0.00,1057,12
InvoiceDate,datetime64[us],0,0.00,47635,2009-12-01 07:45:00
Price,float64,0,0.00,2807,6.95
Country,str,0,0.00,43,United Kingdom


,column,n_disguised,examples
0,Description,342,"[damaged, ?, check, ??]"
1,Country,756,[Unspecified]


In [2]:
# 2. Kiểm tra trước làm sạch
print(f"Số dòng trùng hoàn toàn (dữ liệu gốc): {df_raw.duplicated().sum():,}")
print(f"Số dòng thiếu Customer ID: {df_raw['Customer ID'].isna().sum():,}")
print(f"Phạm vi ngày: {df_raw['InvoiceDate'].min()} - {df_raw['InvoiceDate'].max()}")
# Chưa đủ thông tin để xác định cơ chế MAR/MNAR hoặc nguồn gốc từng dòng trùng.

Số dòng trùng hoàn toàn (dữ liệu gốc): 34,335
Số dòng thiếu Customer ID: 243,007
Phạm vi ngày: 2009-12-01 07:45:00 - 2011-12-09 12:50:00


In [3]:
# 3. Làm sạch bằng đúng các hàm trong pipeline (khởi tạo lại log mỗi lần chạy)
df, base_log = clean_transactions(df_raw)
print(f"Sau làm sạch: {len(df):,} dòng")
display(base_log)

Sau làm sạch: 794,437 dòng


,Step,Column,Finding,Action,Rows,Why
0,1.1,all_columns,"Tên cột chưa đồng nhất, InvoiceDate dạng chuỗi","Chuẩn hóa snake_case, ép kiểu datetime và loại...",1067371,Đảm bảo tính nhất quán khi trích xuất thời gia...
1,1.2,all,"34,335 dòng trùng lặp hoàn toàn","Xóa các dòng trùng lặp, giữ lại bản ghi đầu tiên",34335,Giả định các dòng giống hệt là trùng bản ghi; ...
2,1.3,customer_id,"235,151 dòng thiếu mã khách hàng",Loại bỏ các giao dịch không định danh được khá...,235151,Đơn vị phân tích là hành vi từng khách hàng
3,1.4,description,0 dòng thiếu hoặc chứa ký tự lỗi,Gán nhãn 'UNKNOWN_PRODUCT' thay vì xóa dòng,0,Giữ lại lịch sử chi tiêu và tần suất đơn hàng
4,1.5,country,521 dòng có giá trị 'Unspecified',Điền theo quốc gia phổ biến nhất của cùng cust...,521,Khôi phục thông tin địa lý từ các đơn hàng khá...
5,1.6,stockcode,"3,387 dòng phí vận chuyển/điều chỉnh kế toán",Loại bỏ khỏi bảng giao dịch sản phẩm,3387,Không phản ánh hành vi chọn mua sản phẩm
6,1.7,invoice,"17,593 dòng đơn hủy/hoàn trả",Giữ nguyên và gắn cờ is_cancelled=True,17593,Tín hiệu quan trọng phản ánh hành vi hoàn đơn
7,1.8,price & quantity,61 dòng giá <= 0 hoặc mâu thuẫn dấu số lượng,Loại bỏ bản ghi bất khả thi,61,Tránh làm sai lệch doanh thu và giá trị giỏ hàng


In [4]:
# 4. Chẩn đoán ngoại lai theo IQR; không xóa chỉ vì nằm ngoài ngưỡng
# Tách mua và hoàn trả vì dấu total_amount khác nhau.
outlier_rows = []
for label, group in df.groupby("is_cancelled"):
    q1, q3 = group["total_amount"].quantile([0.25, 0.75])
    iqr = q3 - q1
    count = ((group["total_amount"] < q1 - 1.5 * iqr) | (group["total_amount"] > q3 + 1.5 * iqr)).sum()
    outlier_rows.append({"Step": "IQR_cancel" if label else "IQR_purchase", "Column": "total_amount",
                         "Finding": f"{count:,} dòng ngoài 1.5*IQR", "Action": "Giữ lại; kiểm tra theo nghiệp vụ",
                         "Rows": int(count), "Why": "Ngoại lai thống kê không nhất thiết là lỗi hoặc khách mua buôn"})
log_df = pd.concat([base_log, pd.DataFrame(outlier_rows)], ignore_index=True)
display(pd.DataFrame(outlier_rows))

,Step,Column,Finding,Action,Rows,Why
0,IQR_purchase,total_amount,"62,442 dòng ngoài 1.5*IQR",Giữ lại; kiểm tra theo nghiệp vụ,62442,Ngoại lai thống kê không nhất thiết là lỗi hoặ...
1,IQR_cancel,total_amount,"1,854 dòng ngoài 1.5*IQR",Giữ lại; kiểm tra theo nghiệp vụ,1854,Ngoại lai thống kê không nhất thiết là lỗi hoặ...


In [5]:
# 5. Tái kiểm định và lưu dữ liệu W4
assert df["customer_id"].notna().all()
assert np.isfinite(df[["price", "quantity", "total_amount"]].to_numpy()).all()
assert (df["price"] > 0).all()
assert (df.loc[~df["is_cancelled"], "quantity"] > 0).all()
assert (df.loc[df["is_cancelled"], "quantity"] < 0).all()
assert np.allclose(df["total_amount"], df["price"] * df["quantity"])
assert log_df["Step"].is_unique
print("=== TÁI KIỂM ĐỊNH SAU LÀM SẠCH ===")
display(audit(df))
display(check_disguised_missing(df))
display(log_df)
(ROOT / "reports").mkdir(exist_ok=True)
(ROOT / "data/processed").mkdir(parents=True, exist_ok=True)
log_df.to_csv(ROOT / "reports/cleaning_log_w4.csv", index=False)
df.to_parquet(ROOT / "data/processed/retail_cleaned_w4.parquet", index=False)
print(f"Số dòng sạch: {len(df):,}; chạy python -m src.pipeline ở thư mục gốc trước notebook 02/03.")

=== TÁI KIỂM ĐỊNH SAU LÀM SẠCH ===


,dtype,n_missing,pct_missing,n_unique,sample
invoice,str,0,0.0,43887,489434
stockcode,str,0,0.0,4636,85048
description,str,0,0.0,5246,15CM CHRISTMAS GLASS BALL 20 LIGHTS
quantity,int64,0,0.0,639,12
price,float64,0,0.0,477,6.95
customer_id,str,0,0.0,5876,13085
country,str,0,0.0,41,United Kingdom
invoice_date,datetime64[us],0,0.0,40630,2009-12-01 07:45:00
country_original,str,0,0.0,41,United Kingdom
is_cancelled,bool,0,0.0,2,False


,column,n_disguised,examples
0,country,519,[Unspecified]
1,country_original,519,[Unspecified]


,Step,Column,Finding,Action,Rows,Why
0,1.1,all_columns,"Tên cột chưa đồng nhất, InvoiceDate dạng chuỗi","Chuẩn hóa snake_case, ép kiểu datetime và loại...",1067371,Đảm bảo tính nhất quán khi trích xuất thời gia...
1,1.2,all,"34,335 dòng trùng lặp hoàn toàn","Xóa các dòng trùng lặp, giữ lại bản ghi đầu tiên",34335,Giả định các dòng giống hệt là trùng bản ghi; ...
2,1.3,customer_id,"235,151 dòng thiếu mã khách hàng",Loại bỏ các giao dịch không định danh được khá...,235151,Đơn vị phân tích là hành vi từng khách hàng
3,1.4,description,0 dòng thiếu hoặc chứa ký tự lỗi,Gán nhãn 'UNKNOWN_PRODUCT' thay vì xóa dòng,0,Giữ lại lịch sử chi tiêu và tần suất đơn hàng
4,1.5,country,521 dòng có giá trị 'Unspecified',Điền theo quốc gia phổ biến nhất của cùng cust...,521,Khôi phục thông tin địa lý từ các đơn hàng khá...
5,1.6,stockcode,"3,387 dòng phí vận chuyển/điều chỉnh kế toán",Loại bỏ khỏi bảng giao dịch sản phẩm,3387,Không phản ánh hành vi chọn mua sản phẩm
6,1.7,invoice,"17,593 dòng đơn hủy/hoàn trả",Giữ nguyên và gắn cờ is_cancelled=True,17593,Tín hiệu quan trọng phản ánh hành vi hoàn đơn
7,1.8,price & quantity,61 dòng giá <= 0 hoặc mâu thuẫn dấu số lượng,Loại bỏ bản ghi bất khả thi,61,Tránh làm sai lệch doanh thu và giá trị giỏ hàng
8,IQR_purchase,total_amount,"62,442 dòng ngoài 1.5*IQR",Giữ lại; kiểm tra theo nghiệp vụ,62442,Ngoại lai thống kê không nhất thiết là lỗi hoặ...
9,IQR_cancel,total_amount,"1,854 dòng ngoài 1.5*IQR",Giữ lại; kiểm tra theo nghiệp vụ,1854,Ngoại lai thống kê không nhất thiết là lỗi hoặ...


Số dòng sạch: 794,437; chạy python -m src.pipeline ở thư mục gốc trước notebook 02/03.
